# 03 — Centroid Computation & Rule-Based Action Sequencing

**Goal:** Compute cluster centroids for fast inference, then build a rule-based system for generating structured maintenance action sequences.

## 1. Load Labeled Dataset

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import joblib

df = pd.read_csv('../data/cleaned/dataset3_labeled.csv')
print(f'Loaded {len(df)} labeled records')

# Re-generate embeddings (or load pre-computed)
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['cleaned_description'].tolist(), show_progress_bar=True, batch_size=32)
print(f'Embeddings shape: {embeddings.shape}')

## 2. Compute Cluster Centroids
AgglomerativeClustering doesn't store centroids (unlike K-Means), so we compute them manually.

**Why centroids?** For inference, we compare new embeddings to 7 centroids (fast) vs all 5000 training embeddings (slow).

In [ ]:
centroids = []
for cluster_id in range(7):
    cluster_embeddings = embeddings[df['cluster'] == cluster_id]
    centroid = cluster_embeddings.mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print(f'Centroids shape: {centroids.shape}')  # Expected: (7, 384)

## 3. Save Centroids
These centroids are loaded in the inference pipeline to assign topics to new maintenance records.

In [ ]:
import os
os.makedirs('../models/classifiers', exist_ok=True)
joblib.dump(centroids, '../models/classifiers/topic_centroids.pkl')
print('✅ Centroids saved to models/classifiers/topic_centroids.pkl')

## 4. Rule-Based Action Extraction
**Why rule-based instead of learned?**
1. **Safety-critical domain:** Oil & gas requires validated procedures
2. **Regulatory compliance:** Actions must follow company standards
3. **Interpretability:** Engineers need to understand WHY
4. **Hybrid approach:** ML discovers patterns → rules apply expert-validated responses

In [ ]:
def extract_actions(topic, description):
    """
    Generate structured action recommendations based on topic and keywords.
    Actions follow temporal/logical order: Isolate → Inspect → Repair → Test
    """
    actions = []
    desc_lower = description.lower()

    if topic == 'Pump/Mechanical Failures':
        if 'leak' in desc_lower:
            actions = ['Isolate pump', 'Inspect mechanical seal', 'Replace seal if damaged', 'Pressure test']
        elif 'vibration' in desc_lower:
            actions = ['Check alignment', 'Inspect bearings', 'Balance impeller', 'Tighten foundation bolts']
        else:
            actions = ['Inspect pump', 'Check mechanical components', 'Perform necessary repairs']

    elif topic == 'Valve Operations':
        if 'stuck' in desc_lower or 'seized' in desc_lower:
            actions = ['Apply penetrating oil', 'Work valve manually', 'Disassemble if necessary', 'Lubricate']
        else:
            actions = ['Inspect valve', 'Check actuator', 'Test operation', 'Calibrate as needed']

    elif topic == 'Leakage/Corrosion':
        actions = ['Isolate affected area', 'Assess corrosion extent', 'Clean surface', 'Apply protective coating', 'Monitor regularly']

    elif topic == 'Electrical/Instrumentation':
        actions = ['Verify power supply', 'Inspect wiring connections', 'Test signal integrity', 'Calibrate instrument', 'Replace if faulty']

    elif topic == 'Piping/Structural':
        actions = ['Inspect pipe support condition', 'Check for misalignment', 'Verify pipe wall thickness', 'Reinforce as needed']

    elif topic == 'Operational Issues':
        actions = ['Review operating parameters', 'Check process conditions', 'Adjust settings if required', 'Monitor performance']

    elif topic == 'Preventive Maintenance':
        actions = ['Follow scheduled inspection', 'Perform preventive tasks', 'Replace worn components', 'Update maintenance records']

    else:
        actions = ['Inspect equipment', 'Assess condition', 'Determine required action', 'Report findings']

    return ' → '.join(actions)

# Apply to dataset
df['actions'] = df.apply(lambda row: extract_actions(row['topic'], row['cleaned_description']), axis=1)
print('✅ Actions generated for all records')

## 5. Validate Action Coverage

In [ ]:
print('Action coverage:')
print(df['actions'].apply(lambda x: len(x.split(' → '))).describe())

## 6. Save Dataset with Actions
Ready for FLAN-T5 fine-tuning.

In [ ]:
df.to_csv('../data/cleaned/dataset3_training_ready.csv', index=False)
print(f'✅ Training-ready dataset saved: {df.shape[0]} rows')
print('\nReady for Notebook 04 — FLAN-T5 Fine-Tuning')